In [1]:
from google.colab import drive
drive.mount('/content/drive')
import shutil

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import shutil
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Download
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    shutil.move("/content/UCI HAR Dataset", DATA_PATH)

    print("Download completed")

# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    return X, y


def load_subjects(split):
    """샘플별 피험자 ID 로드 (X, y와 동일한 순서로 대응됨)"""
    path = f"{DATA_PATH}/{split}/subject_{split}.txt"
    return np.loadtxt(path).astype(int)


# Official train/test split
X_train, y_train = load_data("train")
X_test, y_test = load_data("test")

# 피험자 ID (subject 기반 val 분리에 사용)
subject_train = load_subjects("train")


print("\nDataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("subject_train:", subject_train.shape)

Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)
subject_train: (7352,)


In [3]:
# ============================================================
# 5. Normalization
# ============================================================

mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)

In [4]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)

In [5]:
from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# 1. Group K-fold 설정 (피험자 단위로 겹치지 않게 분할)
# ------------------------------------------------------------

N_FOLDS = 5

gkf_full = GroupKFold(n_splits=N_FOLDS)

class_weight_full = {
    0: 1.0,   # WALKING
    1: 1.0,   # WALKING_UPSTAIRS
    2: 1.0,   # WALKING_DOWNSTAIRS
    3: 2.0,   # SITTING
    4: 1.0,   # STANDING
    5: 1.0,   # LAYING
}

class_names = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
               "SITTING", "STANDING", "LAYING"]

fold_models_full = []
fold_val_accuracies_full = []
test_probs_full = []   # 각 fold 모델의 X_test 예측 확률을 저장 (나중에 평균)


# ------------------------------------------------------------
# 2. Fold별 학습
# ------------------------------------------------------------

for fold_idx, (train_idx, val_idx) in enumerate(
    gkf_full.split(X_train, y_train, groups=subject_train)
):

    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    X_train_fold = X_train[train_idx]
    y_train_fold = y_train_onehot[train_idx]

    X_val_fold = X_train[val_idx]
    y_val_fold = y_train_onehot[val_idx]

    print(f"Train subjects: {len(np.unique(subject_train[train_idx]))}, "
          f"Val subjects: {len(np.unique(subject_train[val_idx]))}")
    print(f"X_train_fold: {X_train_fold.shape}, X_val_fold: {X_val_fold.shape}")

    # 모델 정의 (기존 구조 그대로 재사용)
    inputs_full = Input(shape=(128, 9))
    conv1_full = Conv1D(filters=32, kernel_size=5, activation="relu")(inputs_full)
    pool1_full = MaxPooling1D(pool_size=2)(conv1_full)
    conv2_full = Conv1D(filters=64, kernel_size=3, activation="relu")(pool1_full)
    pool2_full = MaxPooling1D(pool_size=2)(conv2_full)
    flat_full = Flatten()(pool2_full)
    drop1_full = Dropout(0.5)(flat_full)
    dense_full = Dense(128, activation="relu", kernel_regularizer=l2(0.009))(drop1_full)
    drop2_full = Dropout(0.3)(dense_full)
    outputs_full = Dense(6, activation="softmax")(drop2_full)

    model_fold = Model(inputs=inputs_full, outputs=outputs_full)

    model_fold.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    history_fold = model_fold.fit(
        X_train_fold,
        y_train_fold,
        epochs=20,
        batch_size=64,
        validation_data=(X_val_fold, y_val_fold),
        shuffle=True,
        class_weight=class_weight_full,
        verbose=1
    )

    fold_models_full.append(model_fold)
    fold_val_accuracies_full.append(history_fold.history["val_accuracy"][-1])

    # 이 fold 모델로 X_test 예측 확률 저장 (나중에 앙상블 평균용)
    test_probs_full.append(model_fold.predict(X_test, verbose=0))


print("\n" + "=" * 60)
print("FOLD별 최종 val_accuracy:", [f"{a:.4f}" for a in fold_val_accuracies_full])
print("평균 val_accuracy:", f"{np.mean(fold_val_accuracies_full):.4f}")


# ------------------------------------------------------------
# 3. 앙상블 예측 (5개 fold 모델의 확률 평균)
# ------------------------------------------------------------

y_prob_full = np.mean(test_probs_full, axis=0)
y_pred_full = np.argmax(y_prob_full, axis=1)


# ------------------------------------------------------------
# 4. 평가
# ------------------------------------------------------------

accuracy_full = accuracy_score(y_test, y_pred_full)
precision_full = precision_score(y_test, y_pred_full, average="macro")
recall_full = recall_score(y_test, y_pred_full, average="macro")
f1_full = f1_score(y_test, y_pred_full, average="macro")

print("\n==============================")
print("6-CLASS TEST RESULTS (K-fold Ensemble)")
print("==============================")
print(f"Accuracy  : {accuracy_full:.4f}")
print(f"Precision : {precision_full:.4f}  (macro avg)")
print(f"Recall    : {recall_full:.4f}  (macro avg)")
print(f"F1-score  : {f1_full:.4f}  (macro avg)")

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")
print(classification_report(
    y_test,
    y_pred_full,
    target_names=class_names,
    digits=4
))

print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")
print(confusion_matrix(y_test, y_pred_full))


FOLD 1/5
Train subjects: 17, Val subjects: 4
X_train_fold: (5932, 128, 9), X_val_fold: (1420, 128, 9)
Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 14s 71ms/step - accuracy: 0.7513 - loss: 2.0253 - val_accuracy: 0.8824 - val_loss: 1.0049
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9356 - loss: 0.7226 - val_accuracy: 0.9085 - val_loss: 0.5636
Epoch 3/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9447 - loss: 0.4100 - val_accuracy: 0.8993 - val_loss: 0.4508
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9474 - loss: 0.3029 - val_accuracy: 0.9141 - val_loss: 0.3323
Epoch 5/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9514 - loss: 0.2584 - val_accuracy: 0.9007 - val_loss: 0.3555
Epoch 6/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9506 - loss: 0.2455 - val_accuracy: 0.8915 - val_loss: 0.4813
Epoch 7/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9498 - loss: 0.2498 - val_accuracy: 0.9056 - val_loss: 0.3546
Epoch 8/20
93

# **SITTING/STANDING 재분류**

In [6]:
# ============================================================
# Stage 2: SITTING / STANDING 이진 분류용 train dataset
# ============================================================

# 라벨: SITTING=3, STANDING=4  ->  이진 라벨 SITTING=0, STANDING=1
SS_LABELS = [3, 4]
ss_class_names = ["SITTING", "STANDING"]

# ------------------------------------------------------------
# 1. 라벨 기준으로 샘플 추출 (X_train, y_train 원본은 그대로 유지)
# ------------------------------------------------------------
ss_mask = np.isin(y_train, SS_LABELS)

X_ss_train = X_train[ss_mask]
y_ss_train = (y_train[ss_mask] == 4).astype(int)   # SITTING=0, STANDING=1
subject_ss_train = subject_train[ss_mask]          # 이후 GroupKFold용

# ------------------------------------------------------------
# 2. 추출한 데이터만의 통계로 정규화
#    (test에도 동일한 ss_mean, ss_std를 적용해야 하므로 저장)
# ------------------------------------------------------------
ss_mean = X_ss_train.mean(axis=(0, 1), keepdims=True)
ss_std = X_ss_train.std(axis=(0, 1), keepdims=True)

X_ss_train = (X_ss_train - ss_mean) / (ss_std + 1e-8)

# ------------------------------------------------------------
# 3. One-hot 인코딩 (softmax 2출력 + categorical_crossentropy용)
# ------------------------------------------------------------
y_ss_train_onehot = to_categorical(y_ss_train, num_classes=2)

# ------------------------------------------------------------
# 4. 확인
# ------------------------------------------------------------
print("X_ss_train:", X_ss_train.shape)
print("y_ss_train:", y_ss_train.shape)
print("subjects  :", len(np.unique(subject_ss_train)))
for i, name in enumerate(ss_class_names):
    print(f"{name:9s}: {(y_ss_train == i).sum()}")

print("\n정규화 후 채널별 mean (0 근처):", np.round(X_ss_train.mean(axis=(0, 1)), 4))
print("정규화 후 채널별 std  (1 근처):", np.round(X_ss_train.std(axis=(0, 1)), 4))

X_ss_train: (2660, 128, 9)
y_ss_train: (2660,)
subjects  : 21
SITTING  : 1286
STANDING : 1374

정규화 후 채널별 mean (0 근처): [ 0.  0.  0. -0.  0. -0.  0.  0.  0.]
정규화 후 채널별 std  (1 근처): [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [7]:
# ============================================================
# Stage 2: test dataset 구성
#   - Stage 1(_4 앙상블)이 SITTING(3) / STANDING(4)로 예측한 샘플만 추출
# ============================================================

# ------------------------------------------------------------
# 1. 라우팅 대상 선택 (Stage 1 예측 기준)
# ------------------------------------------------------------
ss_route_idx = np.where(np.isin(y_pred_full, SS_LABELS))[0]   # X_test 기준 원래 인덱스

# ------------------------------------------------------------
# 2. 샘플 추출 + train과 동일한 통계(ss_mean, ss_std)로 정규화
#    (X_test는 _4에서 이미 정규화됨 -> 학습 때와 같은 순서로 한 번 더 적용)
# ------------------------------------------------------------
X_ss_test = X_test[ss_route_idx]
X_ss_test = (X_ss_test - ss_mean) / (ss_std + 1e-8)

# ------------------------------------------------------------
# 3. 정답 라벨 보관 (평가용, 학습/예측에는 사용하지 않음)
# ------------------------------------------------------------
y_ss_test_orig = y_test[ss_route_idx]                  # 원래 6클래스 정답
ss_test_valid = np.isin(y_ss_test_orig, SS_LABELS)     # 실제 SITTING/STANDING인 샘플
y_ss_test = (y_ss_test_orig == 4).astype(int)          # 이진 정답 (SITTING=0, STANDING=1)
                                                       # ss_test_valid가 False인 샘플의 값은 의미 없음

# ------------------------------------------------------------
# 4. 확인
# ------------------------------------------------------------
print("X_ss_test:", X_ss_test.shape)
print(f"라우팅된 샘플 수      : {len(ss_route_idx)}")
print(f"  실제 SITTING/STANDING: {ss_test_valid.sum()}")
print(f"  다른 클래스가 섞임   : {(~ss_test_valid).sum()}")

n_true_ss = np.isin(y_test, SS_LABELS).sum()
print(f"\n전체 test의 실제 SITTING/STANDING: {n_true_ss}")
print(f"  Stage 1이 다른 클래스로 보내 누락된 수: {n_true_ss - ss_test_valid.sum()}")

print("\n라우팅된 샘플의 실제 라벨 분포 (Stage 1 예측 -> 정답):")
for pred_label in SS_LABELS:
    sel = y_pred_full[ss_route_idx] == pred_label
    counts = np.bincount(y_ss_test_orig[sel], minlength=6)
    print(f"  예측 {class_names[pred_label]:9s}: "
          + ", ".join(f"{class_names[i]}={c}" for i, c in enumerate(counts) if c > 0))

X_ss_test: (1015, 128, 9)
라우팅된 샘플 수      : 1015
  실제 SITTING/STANDING: 1015
  다른 클래스가 섞임   : 0

전체 test의 실제 SITTING/STANDING: 1023
  Stage 1이 다른 클래스로 보내 누락된 수: 8

라우팅된 샘플의 실제 라벨 분포 (Stage 1 예측 -> 정답):
  예측 SITTING  : SITTING=419, STANDING=78
  예측 STANDING : SITTING=64, STANDING=454


In [8]:
# ============================================================
# Stage 2 모델 정의 (기본 1D-CNN) - 모델을 바꿀 때 이 셀만 수정
# ============================================================

def build_ss_model():
    inputs = Input(shape=(128, 9))

    x = Conv1D(filters=32, kernel_size=3, activation="relu")(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(filters=64, kernel_size=3, activation="relu")(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Flatten()(x)
    x = Dense(128, activation="relu")(x)
    outputs = Dense(2, activation="softmax")(x)      # SITTING / STANDING

    model = Model(inputs=inputs, outputs=outputs)

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    return model

In [9]:
# ============================================================
# Stage 2 학습: 피험자 단위 GroupKFold 5-fold
# ============================================================
from sklearn.model_selection import GroupKFold

SS_EPOCHS = 20
SS_BATCH = 64
SS_N_FOLDS = 5

tf.keras.utils.set_random_seed(SEED)   # 셀 재실행 시 같은 조건에서 시작

ss_gkf = GroupKFold(n_splits=SS_N_FOLDS)

ss_fold_models = []        # 학습된 fold 모델 (이후 test 예측에 사용)
ss_fold_val_acc = []       # fold별 최종 epoch val_accuracy
ss_histories = []          # fold별 학습 곡선

for fold_idx, (tr_idx, va_idx) in enumerate(
    ss_gkf.split(X_ss_train, y_ss_train, groups=subject_ss_train)
):
    print(f"\n{'='*60}")
    print(f"[Stage 2] FOLD {fold_idx + 1}/{SS_N_FOLDS}")
    print(f"{'='*60}")

    print(f"Train subjects: {len(np.unique(subject_ss_train[tr_idx]))}, "
          f"Val subjects: {len(np.unique(subject_ss_train[va_idx]))}")
    print(f"X_train_fold: {X_ss_train[tr_idx].shape}, "
          f"X_val_fold: {X_ss_train[va_idx].shape}")

    model_ss = build_ss_model()

    history_ss = model_ss.fit(
        X_ss_train[tr_idx],
        y_ss_train_onehot[tr_idx],
        epochs=SS_EPOCHS,
        batch_size=SS_BATCH,
        validation_data=(X_ss_train[va_idx], y_ss_train_onehot[va_idx]),
        shuffle=True,
        verbose=1
    )

    ss_fold_models.append(model_ss)
    ss_fold_val_acc.append(history_ss.history["val_accuracy"][-1])
    ss_histories.append(history_ss.history)

print("\n" + "=" * 60)
print("[Stage 2] FOLD별 최종 val_accuracy:", [f"{a:.4f}" for a in ss_fold_val_acc])
print(f"[Stage 2] 평균 val_accuracy: {np.mean(ss_fold_val_acc):.4f} "
      f"(std {np.std(ss_fold_val_acc):.4f})")


[Stage 2] FOLD 1/5
Train subjects: 17, Val subjects: 4
X_train_fold: (2149, 128, 9), X_val_fold: (511, 128, 9)
Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 9s 141ms/step - accuracy: 0.8399 - loss: 0.3676 - val_accuracy: 0.9413 - val_loss: 0.2196
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8804 - loss: 0.2741 - val_accuracy: 0.9354 - val_loss: 0.2231
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9027 - loss: 0.2283 - val_accuracy: 0.9432 - val_loss: 0.2319
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9218 - loss: 0.1897 - val_accuracy: 0.9472 - val_loss: 0.2498
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9409 - loss: 0.1504 - val_accuracy: 0.9452 - val_loss: 0.2773
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9577 - loss: 0.1117 - val_accuracy: 0.9374 - val_loss: 0.2805
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9586 - loss: 0.1171 - val_accuracy: 0.9491 - val_loss: 0.2751
Epoc

In [10]:
# ============================================================
# Stage 2: test 예측 (5-fold 앙상블) + 재분류 전/후 성능 비교
#   - 재분류 전 : Stage 1(_4)이 SITTING/STANDING 중 어느 쪽으로 봤는지
#   - 재분류 후 : Stage 2 앙상블의 예측
#   - 평가 대상 : 라우팅된 샘플 중 실제 SITTING/STANDING인 샘플
# ============================================================

# ------------------------------------------------------------
# 1. Stage 2 앙상블 예측 (fold 모델 확률 평균)
# ------------------------------------------------------------
ss_test_probs = [m.predict(X_ss_test, verbose=0) for m in ss_fold_models]
ss_prob_test = np.mean(ss_test_probs, axis=0)
ss_pred_test = np.argmax(ss_prob_test, axis=1)      # 0: SITTING, 1: STANDING

# ------------------------------------------------------------
# 2. 재분류 전 예측 (Stage 1 예측을 이진 라벨로 변환)
# ------------------------------------------------------------
ss_pred_before = (y_pred_full[ss_route_idx] == 4).astype(int)

# ------------------------------------------------------------
# 3. 평가 함수
# ------------------------------------------------------------
def print_binary_eval(title, y_true, y_pred):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print(f"F1-score  : {f1_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print("\nCLASSIFICATION REPORT")
    print(classification_report(y_true, y_pred, target_names=ss_class_names, digits=4))
    print("CONFUSION MATRIX (행: 정답, 열: 예측)")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

# ------------------------------------------------------------
# 4. 평가 (정답이 SITTING/STANDING인 샘플만)
# ------------------------------------------------------------
v = ss_test_valid

print_binary_eval("[재분류 전] Stage 1 예측 (1015개 라우팅 샘플 기준)",
                  y_ss_test[v], ss_pred_before[v])

print_binary_eval("[재분류 후] Stage 2 앙상블 예측",
                  y_ss_test[v], ss_pred_test[v])

# ------------------------------------------------------------
# 5. 요약 비교
# ------------------------------------------------------------
acc_before = accuracy_score(y_ss_test[v], ss_pred_before[v])
acc_after = accuracy_score(y_ss_test[v], ss_pred_test[v])

print("\n" + "=" * 50)
print("요약")
print("=" * 50)
print(f"재분류 전 정확도: {acc_before:.4f}  ({int((ss_pred_before[v] == y_ss_test[v]).sum())}/{int(v.sum())})")
print(f"재분류 후 정확도: {acc_after:.4f}  ({int((ss_pred_test[v] == y_ss_test[v]).sum())}/{int(v.sum())})")
print(f"변화            : {acc_after - acc_before:+.4f}")

# fold 모델 개별 test 정확도 (앙상블 효과/편차 확인용)
fold_test_acc = [accuracy_score(y_ss_test[v], np.argmax(p, axis=1)[v]) for p in ss_test_probs]
print("\nfold 모델별 test 정확도:", [f"{a:.4f}" for a in fold_test_acc])


[재분류 전] Stage 1 예측 (1015개 라우팅 샘플 기준)
Accuracy  : 0.8601
Precision : 0.8598  (macro avg)
Recall    : 0.8604  (macro avg)
F1-score  : 0.8599  (macro avg)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

     SITTING     0.8431    0.8675    0.8551       483
    STANDING     0.8764    0.8534    0.8648       532

    accuracy                         0.8601      1015
   macro avg     0.8598    0.8604    0.8599      1015
weighted avg     0.8606    0.8601    0.8602      1015

CONFUSION MATRIX (행: 정답, 열: 예측)
[[419  64]
 [ 78 454]]

[재분류 후] Stage 2 앙상블 예측
Accuracy  : 0.9034
Precision : 0.9080  (macro avg)
Recall    : 0.9010  (macro avg)
F1-score  : 0.9026  (macro avg)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

     SITTING     0.9405    0.8509    0.8935       483
    STANDING     0.8754    0.9511    0.9117       532

    accuracy                         0.9034      1015
   macro avg     0.9080    0.9010    0.9026      1015
weighted

In [13]:
# ============================================================
# 최종 평가: Stage 2 결과를 Stage 1 예측에 반영한 6클래스 성능
#   - 라우팅된 샘플(ss_route_idx)의 예측만 Stage 2 결과로 덮어씀
#   - 원본 y_pred_full은 그대로 유지 (셀 반복 실행해도 안전)
# ============================================================

# ------------------------------------------------------------
# 1. 예측 결합 (0 -> SITTING(3), 1 -> STANDING(4))
# ------------------------------------------------------------
y_pred_final = y_pred_full.copy()
y_pred_final[ss_route_idx] = np.array(SS_LABELS)[ss_pred_test]

# ------------------------------------------------------------
# 2. 평가 함수
# ------------------------------------------------------------
def print_6class_eval(title, y_true, y_pred):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print(f"F1-score  : {f1_score(y_true, y_pred, average='macro'):.4f}  (macro avg)")
    print("\nCLASSIFICATION REPORT")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    print("CONFUSION MATRIX (행: 정답, 열: 예측)")
    print(confusion_matrix(y_true, y_pred))

# ------------------------------------------------------------
# 3. 결합 전(Stage 1 단독) / 결합 후(Stage 1 + Stage 2) 출력
# ------------------------------------------------------------
print_6class_eval("[결합 전] Stage 1 단독 (6-class)", y_test, y_pred_full)
print_6class_eval("[결합 후] Stage 1 + Stage 2 (6-class)", y_test, y_pred_final)

# ------------------------------------------------------------
# 4. 요약 비교
# ------------------------------------------------------------
cm_before = confusion_matrix(y_test, y_pred_full)
cm_after = confusion_matrix(y_test, y_pred_final)

print("\n" + "=" * 60)
print("요약")
print("=" * 60)
for name, fn in [("Accuracy ", accuracy_score),
                 ("Precision", lambda a, b: precision_score(a, b, average="macro")),
                 ("Recall   ", lambda a, b: recall_score(a, b, average="macro")),
                 ("F1-score ", lambda a, b: f1_score(a, b, average="macro"))]:
    b, a = fn(y_test, y_pred_full), fn(y_test, y_pred_final)
    print(f"{name}: {b:.4f} -> {a:.4f}  ({a - b:+.4f})")

print(f"\n오분류 수: {(y_test != y_pred_full).sum()} -> {(y_test != y_pred_final).sum()}")
print("\nCONFUSION MATRIX 변화 (결합 후 - 결합 전)")
print(cm_after - cm_before)


[결합 전] Stage 1 단독 (6-class)
Accuracy  : 0.9386
Precision : 0.9390  (macro avg)
Recall    : 0.9402  (macro avg)
F1-score  : 0.9393  (macro avg)

CLASSIFICATION REPORT
                    precision    recall  f1-score   support

           WALKING     1.0000    0.9899    0.9949       496
  WALKING_UPSTAIRS     0.9889    0.9469    0.9675       471
WALKING_DOWNSTAIRS     0.9332    0.9976    0.9643       420
           SITTING     0.8431    0.8534    0.8482       491
          STANDING     0.8764    0.8534    0.8648       532
            LAYING     0.9926    1.0000    0.9963       537

          accuracy                         0.9386      2947
         macro avg     0.9390    0.9402    0.9393      2947
      weighted avg     0.9389    0.9386    0.9385      2947

CONFUSION MATRIX (행: 정답, 열: 예측)
[[491   0   5   0   0   0]
 [  0 446  25   0   0   0]
 [  0   1 419   0   0   0]
 [  0   4   0 419  64   4]
 [  0   0   0  78 454   0]
 [  0   0   0   0   0 537]]

[결합 후] Stage 1 + Stage 2 (6-class)